In [ ]:
import os
# Work around duplicate OpenMP runtime (conda numpy + pip torch both ship libomp.dylib).
# Must be set BEFORE importing torch/numpy, otherwise the kernel aborts (OMP: Error #15).
os.environ["KMP_DUPLICATE_LIB_OK"] = "TRUE"

import sys
# Kernel runs from src/; add the repo root so `from src.augmentations import ...` resolves.
sys.path.insert(0, '..')

import torch
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from PIL import Image
from torchvision import transforms
from src.augmentations import AUGMENTATIONS

In [ ]:
# Load a sample image from the dataset CSV
CSV_PATH = '../data/dataset.csv'
df = pd.read_csv(CSV_PATH)
sample_path = df[df['split'] == 'train']['abs_path'].iloc[0]
print(f"Using: {sample_path}")

to_tensor = transforms.Compose([
    transforms.Resize((96, 96)),
    transforms.Grayscale(num_output_channels=3),
    transforms.ToTensor(),
    # transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]) # Paper didnt mention doing ImageNet normalisation
])

original = to_tensor(Image.open(sample_path).convert('RGB'))

In [ ]:
from matplotlib.patches import Rectangle

BBOX_AUGS = {'patch', 'random_erasing'}

def show(tensor):
    return tensor.permute(1, 2, 0).numpy().clip(0, 1)

def draw_bbox(ax, original, augmented):
    diff = (original - augmented).abs().sum(0) > 0
    rows = diff.any(1).nonzero(as_tuple=False)
    cols = diff.any(0).nonzero(as_tuple=False)
    if len(rows) and len(cols):
        y0, y1 = rows[0].item(), rows[-1].item()
        x0, x1 = cols[0].item(), cols[-1].item()
        ax.add_patch(Rectangle((x0, y0), x1 - x0, y1 - y0,
                                linewidth=2, edgecolor='red', facecolor='none'))

aug_names = list(AUGMENTATIONS.keys())
n = len(aug_names)

fig, axes = plt.subplots(1, n + 1, figsize=(4 * (n + 1), 4))

axes[0].imshow(show(original))
axes[0].set_title('Original')
axes[0].axis('off')

for i, name in enumerate(aug_names):
    aug = AUGMENTATIONS[name]()
    augmented = aug(original.clone())
    axes[i + 1].imshow(show(augmented))
    axes[i + 1].set_title(name)
    axes[i + 1].axis('off')
    if name in BBOX_AUGS:
        draw_bbox(axes[i + 1], original, augmented)

plt.tight_layout()
plt.show()

In [ ]:
# Show multiple samples per augmentation
N_SAMPLES = 4
sample_paths = df[df['split'] == 'train']['abs_path'].iloc[:N_SAMPLES].tolist()

for name in aug_names:
    aug = AUGMENTATIONS[name]()
    fig, axes = plt.subplots(1, N_SAMPLES, figsize=(4 * N_SAMPLES, 4))
    fig.suptitle(name, fontsize=14)
    for j, path in enumerate(sample_paths):
        img = to_tensor(Image.open(path).convert('RGB'))
        augmented = aug(img.clone())
        axes[j].imshow(show(augmented))
        axes[j].axis('off')
        if name in BBOX_AUGS:
            draw_bbox(axes[j], img, augmented)
    plt.tight_layout()
    plt.show()